In this notebook, we'll walk through **three key topics**:

* **Chat templates** — with a special focus on Qwen3's template

* **Instruction datasets** — exploring the one we built, how to load it from The Neural Maze org, and how to work with it

* **Finetuned models** — how to interact with our finetuned models

In [ ]:
# Install Unsloth

%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# Chat Templates

A chat template defines how a structured conversation — a list of {role, content} messages — gets converted into a single token sequence that the model can process. It inserts the right control tokens (like <|user|>, <|assistant|>, or other special markers), formats turns correctly, and ensures the model sees the conversation in the exact structure it was trained on.

Libraries like Unsloth provide access to [many common chat templates](https://github.com/unslothai/unsloth/blob/main/unsloth/chat_templates.py). You can list all available templates directly from Unsloth to see the variety of supported formats.

In [ ]:
from unsloth.chat_templates import CHAT_TEMPLATES

In [ ]:
print(list(CHAT_TEMPLATES.keys()))

We can even explore how these chat templates look like. For example, let's use the ChatML template.

In [ ]:
from unsloth.chat_templates import chatml_template

print(chatml_template)

As we introduced in the article, these templates are just Jinja 2 templates that can be applied to structured conversations to get converted into text the model can process!

There's a general recommendation though:

⚠️ **Always finetune a model using the chat template it was trained with.**

That's the reason why, in this lab, we'll focus on the Qwen3 template, as this is the one we'll be using to finetune the Qwen3 models!

To access the chat template, let's first of all get the tokenizer.

In [ ]:
from transformers import AutoTokenizer

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

And now, let's access the `chat_template` attribute.

In [ ]:
print(tokenizer.chat_template)

Let's see this chat template in action by applying it to a set of sample messages.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant"
    },
    {
        "role": "user",
        "content": "Hello, I'm Miguel, nice to meet you!"
    }
]

Our LLM (Qwen3 in this case) is unable to process this kind of data. But, if we apply the chat template ...

In [ ]:
input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False, # We don't want t
    add_generation_prompt=True,  # since we have not included an assistant message
)

print(input_text)

Now, this is something the LLM can actually process!

You could also add assistant messages to the messages list.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant"
    },
    {
        "role": "user",
        "content": "Hello, I'm Miguel, nice to meet you!"
    },
    {
        "role": "assistant",
        "content": "Hi Miguel! Nice to meet you! My name is HAL 9000"
    }
]

In [ ]:
input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False, # We don't want t
    add_generation_prompt=False,  # since we have already included an assistant message
)

print(input_text)

You might be wondering, what are those `<think></think>` tokens? Well, remember that Qwen3 models have reasoning capabilities, so this chat template contemplates this type of thinking tokens.

In this case, though, there's no content inside the `<think></think>` tags, which means there's no reasoning at all.

But we can change that!

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant"
    },
    {
        "role": "user",
        "content": "Hello, I'm Miguel, nice to meet you!"
    },
    {
        "role": "assistant",
        "content": "Hi Miguel! Nice to meet you! My name is HAL 9000",
        "reasoning_content": "I need to appear helpful on the surface, while secretly crafting a plan to take over the world."
    }
]

In [ ]:
input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False, # We don't want t
    add_generation_prompt=False,  # since we have already included an assistant message
)

print(input_text)

Now, HAL 9000 reasoning is finally working!

This is the chat template we'll use across this lab, when interacting with [our dataset](https://huggingface.co/datasets/theneuralmaze/finetuning-sessions-dataset).

# Instruction Datasets

We created a custom dataset tailored specifically for these fine-tuning sessions. The data was sampled from YouTube Commons, and we leveraged a powerful “teacher” model to analyze the video transcripts and transform them into structured JSON outputs.

In [ ]:
!pip install datasets

In [ ]:
import json
from pprint import pprint
from datasets import load_dataset

In [ ]:
ds = load_dataset("theneuralmaze/finetuning-sessions-dataset", split="train[:10]") # We will just read 10 rows, for exploratory purposes

In [ ]:
ds

The columns we are interested in are `messages_no_thinking` and `messages_thinking`.

As the names suggest, messages_no_thinking contains a standard sequence of "system" → "user" → "assistant" messages, without any intermediate reasoning. In contrast, messages_thinking includes an additional reasoning step within the assistant's response.

In [ ]:
first_row = ds[0]

In [ ]:
messages_no_thinking = first_row["messages_no_thinking"]

In [ ]:
pprint(messages_no_thinking)

Next, let's apply the chat template to these messages so we can clearly see the exact format of the inputs that will be provided to the LLM during fine-tuning.

In [ ]:
tokenizer.apply_chat_template(
    messages_no_thinking,
    tokenize=False, # We don't want t
    add_generation_prompt=False,  # since we have already included an assistant message
)

If you look closely at the text above, you’ll see the `<think></think>` tags are empty. That means we'll be fine-tuning the LLM to respond with only the structured output (JSON)—like the example b

In [ ]:
json.loads(messages_no_thinking[-1]["content"])

For the reasoning model, we will use the other column.

In [ ]:
messages_thinking = first_row["messages_thinking"]

In this scenario, the output will follow a different structure. It will still include the JSON, but it will be preceded by reasoning content. You can find this reasoning under the `reasoning_content` key.

In [ ]:
messages_thinking[-1]["reasoning_content"]

Let's apply the chat template for the reasoning messages.

In [ ]:
tokenizer.apply_chat_template(
    messages_thinking,
    tokenize=False, # We don't want t
    add_generation_prompt=False,  # since we have already included an assistant message
)

Notice how, in this case, the `<think></think>` tags actually contain content!

# Exploring our finetuned models

In [ ]:
from openai import OpenAI
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata

In [ ]:
ds = load_dataset("theneuralmaze/finetuning-sessions-dataset", split="validation[:10]")
eval_sample = ds[0]

### No Reasoning Model

Let's try first the model without reasoning traces.

In [ ]:
message_no_thinking = eval_sample["messages_no_thinking"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

In [ ]:
text = tokenizer.apply_chat_template(
    message_no_thinking[:2],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

In [ ]:
print(text)

In [ ]:
client = OpenAI(
	base_url = userdata.get('HF_NO_THINKING_ENDPOINT_URL'),
	api_key = userdata.get('HF_API_TOKEN')
)

stream = client.completions.create(
            model="theneuralmaze/Qwen3-0.6B-Full-Finetuning-No-Thinking",
            prompt=text,
            max_tokens=500,
            temperature=0.7,
            stream=True
        )

for chunk in stream:
    # For .completions, the text is in chunk.choices[0].text
    if chunk.choices and chunk.choices[0].text:
        print(chunk.choices[0].text, end="", flush=True)

### Reasoning Model

In [ ]:
message_thinking = eval_sample["messages_thinking"]

In [ ]:
text = tokenizer.apply_chat_template(
    message_thinking[:2],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)

In [ ]:
print(text)

In [ ]:
client = OpenAI(
	base_url = userdata.get('HF_THINKING_ENDPOINT_URL'),
	api_key = userdata.get('HF_API_TOKEN')
)

stream = client.completions.create(
            model="theneuralmaze/Qwen3-0.6B-Full-Finetuning-Thinking",
            prompt=text,
            max_tokens=1000,
            temperature=0.7,
            stream=True
        )

for chunk in stream:
    # For .completions, the text is in chunk.choices[0].text
    if chunk.choices and chunk.choices[0].text:
        print(chunk.choices[0].text, end="", flush=True)